In [1]:
import pandas as pd
import pyiron_workflow as pwf
from ase.calculators import emt
from pyiron_workflow_atomistics import engine as engine_mod

from demonstrators import elastic_nodes

In [2]:
engine = engine_mod.ASEEngine(
    EngineInput=None,
    calculator=emt.EMT(),
    working_directory="demo_runs",
    record_interval=100,
)

# Elastic constants

In [3]:
elastic_wf = pwf.node(elastic_nodes.unary_elastic_tensor)

elastic_wf.validate(do_ontology=True)
# Ontology can't handle constants yet

/Users/liamhuber/dev/miniforge3/envs/pyiron12/lib/python3.12/site-packages/networkx/utils/backends.py:551: UserWarning: The hashes produced for directed graphs changed in version v3.5 due to a bugfix to track in and out edges separately (see documentation).
  return self.orig_func(*args, **kwargs)


Type validation for 'unary_elastic_tensor' (valid=False, complete=False):
	unfulfilled edges:
		elastic_constants_0.summary->get_bulk_modulus_0.elastic_summary
		getitem_0.item->tensor_ieee
	subreports:
	Type validation for 'unary_elastic_tensor.elastic_constants_0' (valid=False, complete=False):
		invalid edges:
			engine->with_calc_input_1.engine
		unfulfilled edges:
			with_calc_input_0.output_0->calculate_0.engine
			get_attr_0.attr->generate_mp_deformations_0.structure
			generate_mp_deformations_0.deformed_structures->evaluate_structures_0.structures
			with_calc_input_1.output_0->evaluate_structures_0.engine
			get_attr_0.attr->fit_elastic_tensor_0.structure
			get_attr_0.attr->elastic_constants_summary_0.structure
		subreports:
		constant_0: <NOT PARSEABLE>
	constant_0: <NOT PARSEABLE>
Validation Report
Conforms: True

In [4]:
elastic_run = elastic_wf.run(
    engine=engine.with_working_directory("elastic"),
    symbol="Au",
    relaxation_config=engine_mod.CalcInputMinimize(relax_cell=True)
)

print(f"Bulk modulus: {elastic_run.outputs.bulk_modulus:.2f} GPa")
print("IEEE elastic tensor (GPa):")
pd.options.display.float_format = '{:.1f}'.format
pd.DataFrame(
    elastic_run.outputs.tensor_ieee,
    index=[f"{i+1}" for i in range(6)],
    columns=[f"{j+1}" for j in range(6)],
)

      Step     Time          Energy          fmax
BFGS:    0 14:43:39        0.002606        0.308859
BFGS:    1 14:43:39        0.000032        0.077465
BFGS:    2 14:43:39       -0.000135        0.002603
Bulk modulus: 174.13 GPa
IEEE elastic tensor (GPa):


,1,2,3,4,5,6
1,196.8,162.8,162.8,-0.0,0.0,0.0
2,162.8,196.8,162.8,-0.0,0.0,0.0
3,162.8,162.8,196.8,-0.0,0.0,0.0
4,-0.0,-0.0,-0.0,54.9,0.0,0.0
5,0.0,0.0,0.0,0.0,54.9,-0.0
6,0.0,0.0,0.0,0.0,-0.0,54.9
